In [40]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow import keras
from sklearn.model_selection import train_test_split
import numpy as np



In [ ]:
#Daten vorbereiten
df = pd.read_csv("melting-point-data/train_ext.csv")
y = np.array(df["Tm"]).reshape(-1,1)

df = df.drop(["Unnamed: 0", "Tm"], axis = 1)  



X = np.array(df.drop(columns = df.filter(regex = "Group").columns))
X = np.array(df)


#Daten skalieren
x_scale = StandardScaler()
y_scale = StandardScaler()

pca = PCA(n_components=150)
X_train = pca.fit_transform(X)

X_train = x_scale.fit_transform(X_train)
y_train = y_scale.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42)



In [42]:
#Modell aufbauen

callback = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20)

num_features = X_train.shape[1]

model = Sequential()
model.add(Dense(62, input_dim = num_features, activation = 'relu', kernel_regularizer=l2(0.001) ))
model.add(Dropout(0.2))
model.add(Dense(128, activation = 'sigmoid', kernel_regularizer=l2(0.001) ))
model.add(Dropout(0.2))
model.add(Dense(32, activation = 'relu', kernel_regularizer=l2(0.001) ))

model.add(Dense(1, activation = 'linear' ))

model.summary()


/Users/enricofritz/Documents/Meltingpoint_rep/Melting_point_project/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ (None, 62)             │        27,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 62)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 128)            │         8,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,505 (154.32 KB)

 Trainable params: 39,505 (154.32 KB)

 Non-trainable params: 0 (0.00 B)

In [43]:
model.compile(optimizer=Adam(learning_rate=0.0005), loss = "mae", metrics=["mae"])
history = model.fit(X_train,y_train, verbose = True, validation_split = 0.1, epochs = 250, callbacks=[callback])

Epoch 1/250
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 270.4733 - mae: 270.2715 - val_loss: 273.8583 - val_mae: 273.6893
Epoch 2/250
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 749us/step - loss: 251.5249 - mae: 251.3731 - val_loss: 250.0720 - val_mae: 249.9328
Epoch 3/250
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 714us/step - loss: 223.9281 - mae: 223.7933 - val_loss: 218.7682 - val_mae: 218.6348
Epoch 4/250
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 709us/step - loss: 188.3599 - mae: 188.2228 - val_loss: 178.6257 - val_mae: 178.4828
Epoch 5/250
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 705us/step - loss: 144.7041 - mae: 144.5519 - val_loss: 131.8161 - val_mae: 131.6530
Epoch 6/250
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 707us/step - loss: 102.9683 - mae: 102.7937 - val_loss: 95.1357 - val_mae: 94.9499
Epoch 7/250
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 700us/step - loss: 78.2455 - mae: 78.0508 - val_loss: 76.7893 - val_mae: 76.5865
Epoch 8/250
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 704us/step - loss: 68.3157 - mae: 68.1068 - val_loss: 69.6442 - val_mae: 69.4

In [45]:
y_predict = y_scale.inverse_transform(model.predict(X_test))
y_true = y_scale.inverse_transform(y_test)

mae = np.mean(np.abs(y_predict - y_true))

mae

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 611us/step


np.float64(2922.7865481208564)

KeyError: 'MolWeight'